In [1]:
import duckdb
import polars as pl

from orqa.utils import load_dataframe

In [2]:
queries_path = 'dataset/orqa_v1.csv'

df = pl.read_csv(queries_path)

In [24]:
idx = 542
r_rsc_id = df.row(idx, named=True)['r_rsc_id']
s_rsc_id = df.row(idx, named=True)['s_rsc_id']
sql = df.row(idx, named=True)['sql']
df.row(idx)

('CAN',
 'single-table',
 'challenging',
 True,
 'ed783874-8549-42e6-9d4e-ac223c16617e',
 None,
 '2fe6b7f1-fbfe-4aaf-808f-b977d40d44db',
 None,
 'Staffing activities by type, organization and geographic area',
 None,
 None,
 None,
 'SELECT "i_fiscal_year_", "geo_region_e", SUM("n") AS total_n, RANK() OVER (PARTITION BY "i_fiscal_year_" ORDER BY SUM("n") DESC) as rank FROM R GROUP BY "i_fiscal_year_", "geo_region_e"',
 'Could you rank the geographic regions based on the total number of staffing activities for each fiscal year? This would help us understand which areas had the most hiring and promotion activities under the Public Service Employment Act during different periods.',
 'success',
 6.847,
 0,
 None,
 2055,
 296,
 6.489,
 0,
 None,
 2575,
 158,
 13.336)

In [25]:
sql

'SELECT "i_fiscal_year_", "geo_region_e", SUM("n") AS total_n, RANK() OVER (PARTITION BY "i_fiscal_year_" ORDER BY SUM("n") DESC) as rank FROM R GROUP BY "i_fiscal_year_", "geo_region_e"'

In [37]:
load_dataframe('data/datasets/CAN/tables/from0_toEND/b28b56d5-3d62-45b2-8528-07f4b9005305.parquet', 'complex', None)

iorganization,vote,description,2022_23_expenditures,2023_24_main_estimates,2023_24_estimates_to_date,2024_25_main_estimates
str,str,str,f32,f32,f32,f32
"""Administrative Tribunals Suppo…","""1""","""Program expenditures""",7.0734224e7,6.7956136e7,6.7956136e7,6.8646256e7
"""Administrative Tribunals Suppo…","""S""","""Contributions to employee bene…",1.1861206e7,1.2401922e7,1.2401922e7,1.2012655e7
"""Atlantic Canada Opportunities …","""1""","""Operating expenditures""",7.2473288e7,6.840408e7,7.0675496e7,7.0390768e7
"""Atlantic Canada Opportunities …","""5""","""Grants and contributions""",3.51386496e8,3.12855584e8,4.37992096e8,3.0669488e8
"""Atlantic Canada Opportunities …","""S""","""Contributions to employee bene…",9.264243e6,8.967542e6,9.496339e6,8.557714e6
…,…,…,…,…,…,…
"""VIA HFR - VIA TGF Inc.""","""1""","""Payments to the corporation fo…",0.0,4.367e7,5.167e7,4.5254e7
"""VIA Rail Canada Inc.""","""1""","""Payments to the Corporation""",6.89627776e8,1.2336e9,1.4825e9,1.1593e9
"""Veterans Review and Appeal Boa…","""1""","""Program expenditures""",1.2153646e7,1.3837908e7,1.6539833e7,2.0045176e7


In [26]:
R = load_dataframe(f'data/datasets/{"CAN" if idx >= 500 else "UK"}/tables/from0_toEND/{r_rsc_id}.parquet', 'complex', None)
if s_rsc_id:
    S = load_dataframe(f'data/datasets/{"CAN" if idx >= 500 else "UK"}/tables/from0_toEND/{s_rsc_id}.parquet', 'complex', None)
else:
    S = None

In [27]:
R.head()

i_fiscal_year_,geo_region_e,geo_region_f,orgnization_e,organization_f,staffing_act_appt_type_e,staffing_act_appt_type_f,description_e,description_f,n
str,str,str,str,str,str,str,str,str,i32
"""2015_2016""","""Newfoundland and Labrador""","""Terre-Neuve-et-Labrador""","""Atlantic Canada Opportunities …","""Agence de promotion Ã©conomiqu…","""Promotions""","""Promotions""","""Staffing activities within the…","""ActivitÃ©s de dotation dans la…",6
"""2015_2016""","""Newfoundland and Labrador""","""Terre-Neuve-et-Labrador""","""Atlantic Canada Opportunities …","""Agence de promotion Ã©conomiqu…","""Lateral transfers (same classi…","""Mutations latÃ©rales (mÃªme cl…","""Staffing activities within the…","""ActivitÃ©s de dotation dans la…",21
"""2015_2016""","""Newfoundland and Labrador""","""Terre-Neuve-et-Labrador""","""Atlantic Canada Opportunities …","""Agence de promotion Ã©conomiqu…","""Hiring activity to the Public …","""Nominations externes""","""Appointments to the public ser…","""Nominations Ã la fonction pub…",3
"""2015_2016""","""Newfoundland and Labrador""","""Terre-Neuve-et-Labrador""","""Atlantic Canada Opportunities …","""Agence de promotion Ã©conomiqu…","""Acting appointments""","""Nominations intÃ©rimaires""","""Staffing activities within the…","""ActivitÃ©s de dotation dans la…",2
"""2015_2016""","""Newfoundland and Labrador""","""Terre-Neuve-et-Labrador""","""Agriculture and Agri-Food Cana…","""Agriculture et Agroalimentaire…","""Promotions""","""Promotions""","""Staffing activities within the…","""ActivitÃ©s de dotation dans la…",1


In [29]:
S.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [30]:
R.shape, S.shape if isinstance(S, pl.DataFrame) else None

((11910, 10), None)

In [31]:
#duckdb.rollback()
results = duckdb.sql(sql).fetchmany(size=1)
results

[('2017_2018', 'National Capital Region', 40414, 1)]

In [38]:
import pickle
import bidict

In [39]:
with open('data/datasets/CAN/database/from0_toEND/values_bidict.pickle', 'rb') as file:
    bd = pickle.load(file)

In [67]:
import re

def is_num(x) -> bool:
    "Very very simple solution, but for many cases works"
    if isinstance(x, str):
        x = x.replace(',', '').replace('.', '').replace('%', ' ')
    try: 
        float(x)
    except: 
        return False
    return True

In [40]:
len(bd)

1328331254

In [65]:
re.match(r'[^0-9](\d{1,100})', 'v1432313432 ')

<re.Match object; span=(0, 11), match='v1432313432'>

In [76]:
import random
for i in random.sample(range(1000000000), 10):
    print(bd.inverse.get(i), is_num(bd.inverse.get(i)))

4805.2.12.6.4 True
2689.6.2.3.4 True
69.5.2.9.2.1.7 True
15.2.15.5.5.1.3.17 True
v39624236 False
14.3.13.3.1.98 True
1.1.1.144.2.2.5 True
131.3.3.2.4.1.5 True
v97970712 False
92.4.2.1.519 True
